In [ ]:
from pathlib import Path
from metasmith.python_api import Agent, Source, Std, DataInstanceLibrary, TransformInstanceLibrary, WorkflowTask
from metasmith.python_api import DataTypeLibrary, Endpoint
from local.constants import WORKSPACE_ROOT

# dtypes, containers, transforms = Std()

path_to_agent_home = Path("./cache/local_home").resolve()
smith = Agent(
    home = Source.FromLocal(path_to_agent_home),
)
# smith.Deploy()

In [ ]:
mock_types = DataTypeLibrary(types=dict(
    a=Endpoint({"test", "a"}),
    b=Endpoint({"test", "b"}),
    c=Endpoint({"test", "c"}),
))

transforms = TransformInstanceLibrary("./transforms/simple_1", include_std=False)
transforms.AddTypeLibrary("mock", mock_types)
transforms.AddStub("no_op")
transforms.Save()

In [ ]:
in_path = WORKSPACE_ROOT/"main/local_mock/cache/mock_dt"
with open(in_path, "w") as f:
    f.write("120")
inputs = DataInstanceLibrary("./cache/dev20.mock.xgdb")
inputs.AddTypeLibrary("mock", mock_types)
inputs.AddItem(in_path, "mock::a")
inputs.Save()
for p, n, e in inputs.Iterate():
    print(n, p, e, e.parents)

In [ ]:
for loc, t,  in transforms.IterateTransforms():
    print(t.model)

In [ ]:
N = 100
_tasks = [
    smith.GenerateWorkflow(
        given      = [inputs],
        transforms = [transforms],
        targets    = [mock_types["b"]]
    )
    # for t in ["per_contig_coverage"]
    for _ in range(N)
]
with open(WORKSPACE_ROOT/"secrets/slurm_account_fir") as f:
    SLURM_ACCOUNT = f.read()
task = WorkflowTask.Merge(
    _tasks,
    config=dict(
        nextflow = dict(
            preset="slurm",
            slurm_account=SLURM_ACCOUNT,
            cpus=1,
            queueSize=100,
            array=10,
            memory='32 GB',
            time='6h',
        ),
    )
)
print(task.GetKey(), len([s for p in task.plans for s in p.steps]))
for step in [s for p in task.plans for s in p.steps][:3]:
    print(step.order, step.transform.name)
# task.RenderDAG("./cache/dag")

In [ ]:
smith.StageWorkflow(task, on_exist="clear", verify_external_paths=False)

In [ ]:
smith.RunWorkflow(task)

In [ ]:
smith.CheckWorkflow(task)